# Análise de avaliações de filmes com PySpark (MapReduce)

Projeto acadêmico de Big Data (UNINTER). Usa o dataset `imdb-reviews-pt-br` e o modelo MapReduce com RDDs do PySpark, executado no Google Colab.

## 1. Preparação do ambiente

In [ ]:
!wget https://raw.githubusercontent.com/N-CPUninter/Big_Data/main/data/imdb-reviews-pt-br.zip -O imdb-reviews-pt-br.zip
!unzip imdb-reviews-pt-br.zip
!rm imdb-reviews-pt-br.zip

In [ ]:
!apt-get install -y openjdk-17-jdk-headless
!pip install pyspark

In [ ]:
import setuptools
import os
from pyspark.sql import SparkSession

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] += os.pathsep + os.path.join(os.environ["JAVA_HOME"], "bin")

spark = SparkSession.builder.appName("Analise de avaliacoes PySpark").master("local").getOrCreate()

In [ ]:
imdb_df = spark.read.csv('imdb-reviews-pt-br.csv',
                         header=True,
                         quote="\"",
                         escape="\"",
                         encoding="UTF-8")

## 2. Soma dos IDs das avaliações negativas

**Map:** cada linha vira o par `(sentiment, id)`.  
**Reduce:** soma os IDs de cada sentimento.

In [ ]:
def map1(x):
    return (x['sentiment'], int(x['id']))

def reduceByKey1(x, y):
    return x + y

resultado1 = imdb_df.rdd.map(map1).reduceByKey(reduceByKey1).collect()
print(resultado1)
print("Soma dos ids negativos:", dict(resultado1)['neg'])

## 3. Diferença de palavras (português x inglês) nas avaliações negativas

**Map:** cada linha vira `(sentiment, (palavras_en, palavras_pt))`.  
**Reduce:** soma as duas contagens por sentimento.

In [ ]:
def map2(x):
    return (x['sentiment'], (len(x['text_en'].split()), len(x['text_pt'].split())))

def reduceByKey2(x, y):
    return (x[0] + y[0], x[1] + y[1])

resultado2 = imdb_df.rdd.map(map2).reduceByKey(reduceByKey2).collect()
print(resultado2)

neg_en, neg_pt = dict(resultado2)['neg']
print("Palavras EN (neg):", neg_en)
print("Palavras PT (neg):", neg_pt)
print("Diferença PT - EN:", neg_pt - neg_en)